# EV2Gym simple tutorial

This notebook is a small first showcase of EV2Gym.

The goal is to run one ready-made EV2Gym scenario, use a simple charging rule, and inspect the main outputs. It is not yet the final thesis scenario. It is a practical starting point before creating a custom setup with one EV, one charger, PV production, and variable electricity prices.

## What EV2Gym does

EV2Gym simulates electric vehicle charging. The simulator keeps track of chargers, connected EVs, electricity prices, power limits, and rewards.

A basic EV2Gym workflow is:

1. Load a YAML configuration file.
2. Create the EV2Gym environment.
3. Reset the environment to start a new simulation.
4. Choose an action for every charging port at every time step.
5. Step the environment forward and collect rewards and statistics.

In this notebook, the actions come from a simple built-in baseline: `ChargeAsFastAsPossible`.

In [ ]:
from ev2gym.models.ev2gym_env import EV2Gym
from ev2gym.baselines.heuristics import ChargeAsFastAsPossible

import numpy as np
import matplotlib.pyplot as plt
import yaml
import pprint
import os

## 1. Choose an example configuration

EV2Gym scenarios are defined in YAML files. These files describe the simulation length, number of chargers, EV behavior, prices, PV generation, and other settings.

Here we use an existing example file:

`ev2gym/example_config_files/V2GProfitPlusLoads.yaml`

This file is useful for a first test because it already includes variable prices, inflexible loads, and solar power settings.

In [ ]:
config_file = "ev2gym/example_config_files/V2GProfitPlusLoads.yaml"

print("Current working directory:")
print(os.getcwd())
print("\nConfig file:", config_file)

with open(config_file, "r") as file:
    config = yaml.safe_load(file)

important_config_values = {
    "timescale_minutes": config["timescale"],
    "simulation_length_steps": config["simulation_length"],
    "scenario": config["scenario"],
    "v2g_enabled": config["v2g_enabled"],
    "charging_stations": config["number_of_charging_stations"],
    "ports_per_station": config["number_of_ports_per_cs"],
    "solar_power_enabled": config["solar_power"]["include"],
    "inflexible_loads_enabled": config["inflexible_loads"]["include"],
}

pprint.pp(important_config_values)

## 2. Create the environment

The environment is the simulator object. It contains the charging stations, EVs, prices, power limits, and the current simulation state.

For a tutorial, replay and plot saving are disabled. This keeps the notebook clean. They can be enabled later with `save_replay=True` and `save_plots=True`.

In [ ]:
env = EV2Gym(
    config_file=config_file,
    save_replay=False,
    save_plots=False,
    verbose=False,
    seed=42,
)

state, info = env.reset(seed=42)
env.done = False

print("Environment loaded.")
print("Simulation date:", env.sim_starting_date)
print("Simulation length:", env.simulation_length, "steps")
print("Minutes per step:", env.timescale)
print("Number of charging ports:", env.number_of_ports)
print("Action space:", env.action_space)
print("Observation space:", env.observation_space)

## 3. Look at the first observation

After `reset()`, EV2Gym returns the first observation. This is the information that an algorithm can use to decide what to do next.

The exact content depends on the selected state function. For this first showcase, we only check the type, shape, and a few values.

In [ ]:
print("Observation type:", type(state))
print("Observation shape:", state.shape)
print("First 10 values:")
print(np.round(state[:10], 3))

## 4. Understand actions

An action is one number for each charging port.

Because V2G is enabled in this example, each action is between `-1` and `1`:

- `1` means charge as much as possible.
- `0` means do nothing.
- `-1` means discharge as much as possible.

The simple baseline used here always returns `1` for every port. So it tries to charge every connected EV as fast as possible.

In [ ]:
agent = ChargeAsFastAsPossible()
actions = agent.get_action(env)

print("Action vector shape:", actions.shape)
print("First 10 actions:", actions[:10])
print("Minimum action:", actions.min())
print("Maximum action:", actions.max())

## 5. Run a short simulation

Now we run the simulator with the simple charging baseline.

At every step:

1. The agent chooses actions.
2. EV2Gym applies the actions.
3. The environment returns the next observation, reward, and status.

We store rewards, power usage, and the number of connected ports so we can inspect the result afterwards.

To keep the notebook quick, it runs only the first 24 steps by default. Set `max_steps = env.simulation_length` to run the full simulation.

In [ ]:
state, info = env.reset(seed=42)
env.done = False
agent = ChargeAsFastAsPossible()

rewards = []
power_usage = []
connected_ports = []
last_stats = None

max_steps = min(24, env.simulation_length)

for step in range(max_steps):
    actions = agent.get_action(env)
    state, reward, done, truncated, stats = env.step(actions)

    rewards.append(reward)
    power_usage.append(env.current_power_usage[env.current_step - 1])
    connected_ports.append(stats["action_mask"].sum())
    last_stats = stats

    if done or truncated:
        break

print("Steps run:", len(rewards))
print("Total reward:", round(float(np.sum(rewards)), 2))
print("Current simulation step:", env.current_step)
print("Full simulation finished:", env.done)

## 6. Inspect simple metrics

EV2Gym returns final statistics only when the full simulation is finished.

Because this tutorial uses a short run by default, the cell below works in both cases:

- if the full simulation is finished, it prints final EV2Gym statistics,
- if only a short run was completed, it prints current progress metrics.

In [ ]:
if env.done:
    summary_keys = [
        "total_ev_served",
        "total_profits",
        "total_energy_charged",
        "total_energy_discharged",
        "average_user_satisfaction",
        "energy_tracking_error",
        "total_transformer_overload",
        "battery_degradation",
        "total_reward",
    ]
    summary = {key: last_stats.get(key) for key in summary_keys}
else:
    summary = {
        "steps_run": len(rewards),
        "evs_spawned_so_far": env.total_evs_spawned,
        "evs_currently_parked": env.current_evs_parked,
        "energy_charged_so_far": sum(cs.total_energy_charged for cs in env.charging_stations),
        "energy_discharged_so_far": sum(cs.total_energy_discharged for cs in env.charging_stations),
        "profit_so_far": sum(cs.total_profits for cs in env.charging_stations),
        "reward_so_far": np.sum(rewards),
    }

clean_summary = {}
for key, value in summary.items():
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float):
        value = round(value, 3)
    clean_summary[key] = value

pprint.pp(clean_summary)

## 7. Plot simple results

A plot makes the simulation easier to understand.

The first chart shows the reward at each step. The second chart compares actual charging power with the power setpoint from the environment. The third chart shows how many charging ports had an EV connected during the short run.

In [ ]:
steps = np.arange(len(rewards))
power_usage = np.array(power_usage)
connected_ports = np.array(connected_ports)
power_setpoints = env.power_setpoints[:len(power_usage)]

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

axes[0].plot(steps, rewards)
axes[0].set_ylabel("Reward")
axes[0].set_title("Reward per step")
axes[0].grid(True, alpha=0.3)

axes[1].plot(steps, power_usage, label="Actual power")
axes[1].plot(steps, power_setpoints, label="Power setpoint", linestyle="--")
axes[1].set_ylabel("Power (kW)")
axes[1].set_title("Charging power")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].step(steps, connected_ports, where="post")
axes[2].set_ylabel("Connected ports")
axes[2].set_xlabel("Simulation step")
axes[2].set_title("EVs connected to charging ports")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. What this simple baseline means

`ChargeAsFastAsPossible` is an uncontrolled charging strategy. It does not look ahead and it does not try to optimize prices, PV use, or grid limits. It simply charges every connected EV at maximum power.

This makes it a useful reference case. Later, a smarter strategy can be compared against it. A smart strategy should try to charge at better times while still meeting the EV driver's energy needs.

## 9. Next steps for the thesis scenario

This notebook proves the basic EV2Gym workflow:

- the environment can be loaded,
- the observation and action spaces can be inspected,
- a baseline controller can run the simulator,
- rewards and metrics can be collected,
- simple plots can be created.

The next technical step is to create a smaller custom configuration for the thesis case. That configuration should describe one EV, one charging station, PV production, electricity prices, and the charging strategies that will be compared.